## Introduction

In this notebook, we will fine‐tune a BART model on the SAMSum dataset, which consists of around 1000 pairs of AI generated dialogues of students in a graduate program involving machine learning and their corresponding summaries. By the end, you will have a functioning text summarization model and will see how to evaluate it on held‐out data.

## Setup

Before we start implementing the pipeline, let's install and import all the
libraries we need. We'll be using the KerasHub library. We will also need a
couple of utility libraries.

In [1]:
#!pip install git+https://github.com/keras-team/keras-hub.git py7zr -q
#%pip install py7zr
#%pip install keras_hub
#%pip install jax
#%pip install tensorflow_datasets
#%pip install keras keras-hub


This examples uses [Keras 3](https://keras.io/keras_3/) to work in any of
`"tensorflow"`, `"jax"` or `"torch"`. Support for Keras 3 is baked into
KerasHub, simply change the `"KERAS_BACKEND"` environment variable to select
the backend of your choice. We select the JAX backend below.

In [2]:
#import libraries
import os

os.environ["KERAS_BACKEND"] = "jax"
import py7zr
import time

import keras_hub
import keras
import tensorflow as tf
import tensorflow_datasets as tfds
import json
import tensorflow as tf
from tensorflow.keras import mixed_precision

/Users/benlaufer/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Let's also define our hyperparameters.

In [3]:
# batch size for training
BATCH_SIZE = 8

#set number of patches
NUM_BATCHES = 10

# number of epochs to train
EPOCHS = 5

# Maximum lengths for encoder and decoder sequences.
# We will reduce from the default 1024 to save compute. 
MAX_ENCODER_SEQUENCE_LENGTH = 512
MAX_DECODER_SEQUENCE_LENGTH = 128

# Maximum length of generated summaries at inference time.
MAX_GENERATION_LENGTH = 40


## Dataset

This is the original dataset that was used in the keras project posted:
[SAMSum dataset](https://arxiv.org/abs/1911.12237). 
This dataset contains around 15,000 pairs of conversations/dialogues and summaries.
If you want to use the initial dataset that this model was trained on, it will be pushed to the github

In this project, I will be using a way smaller dataset that contains around 200 pairs of conversations/dialogues and summaries all generated by open AI.

In [4]:
# directory for each json file
train_path = "data/train_new.json"
val_path   = "data/val_new.json"
test_path  = "data/test_new.json"

# read each JSON file into Python lists (raw approach).
# each file is a list of { "dialogue": ..., "summary": ... } entries.
with open(train_path, "r", encoding="utf-8") as f:
    train_records = json.load(f)

with open(val_path, "r", encoding="utf-8") as f:
    val_records = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_records = json.load(f)

# print one raw example
print("Example train record:")
print(train_records[0])


Example train record:
{'id': '30000001', 'summary': 'Alex and Sam discuss grad school.', 'dialogue': 'Alex: Have you started your Cal Poly SLO application?\nSam: Yes, I need to gather my transcripts and letters of recommendation.\nAlex: Remember the deadline is in two weeks.\nSam: I’m also studying for the GRE this weekend.\nAlex: Let’s study together tomorrow.\nSam: Sounds great!'}
Example train record:
{'id': '30000001', 'summary': 'Alex and Sam discuss grad school.', 'dialogue': 'Alex: Have you started your Cal Poly SLO application?\nSam: Yes, I need to gather my transcripts and letters of recommendation.\nAlex: Remember the deadline is in two weeks.\nSam: I’m also studying for the GRE this weekend.\nAlex: Let’s study together tomorrow.\nSam: Sounds great!'}


In [5]:
def load_samsum_split(path_to_json):
    
    """
    Reads a single SAMSum JSON file (a list of { "dialogue":…, "summary":… } entries)
    and returns a tf.data.Dataset of (dialogue, summary) string pairs.
    """

    # open open JSON file and parse it
    with open(path_to_json, "r", encoding="utf-8") as f:
        records = json.load(f)

    # extract the dialogues and summaries into parallel Python lists
    dialogues = [r["dialogue"] for r in records]
    summaries = [r["summary"] for r in records]

    # create a tf.data.Dataset from two parallel lists
    return tf.data.Dataset.from_tensor_slices((dialogues, summaries))


In [6]:
# load each split into a tf.data.Dataset
train_ds = load_samsum_split(train_path)
val_ds   = load_samsum_split(val_path)
test_ds  = load_samsum_split(test_path)

# print one example from the training set
for dialogue, summary in train_ds.take(1):
    print("Dialogue:\n", dialogue.numpy().decode("utf-8"))
    print("\nSummary:\n", summary.numpy().decode("utf-8"))

Dialogue:
 Alex: Have you started your Cal Poly SLO application?
Sam: Yes, I need to gather my transcripts and letters of recommendation.
Alex: Remember the deadline is in two weeks.
Sam: I’m also studying for the GRE this weekend.
Alex: Let’s study together tomorrow.
Sam: Sounds great!

Summary:
 Alex and Sam discuss grad school.
Dialogue:
 Alex: Have you started your Cal Poly SLO application?
Sam: Yes, I need to gather my transcripts and letters of recommendation.
Alex: Remember the deadline is in two weeks.
Sam: I’m also studying for the GRE this weekend.
Alex: Let’s study together tomorrow.
Sam: Sounds great!

Summary:
 Alex and Sam discuss grad school.


The dataset has two fields: dialogue and summary. Let's see a sample.

In [7]:
for dialogue, summary in train_ds:
    print(dialogue.numpy())
    print(summary.numpy())
    break

b'Alex: Have you started your Cal Poly SLO application?\nSam: Yes, I need to gather my transcripts and letters of recommendation.\nAlex: Remember the deadline is in two weeks.\nSam: I\xe2\x80\x99m also studying for the GRE this weekend.\nAlex: Let\xe2\x80\x99s study together tomorrow.\nSam: Sounds great!'
b'Alex and Sam discuss grad school.'
b'Alex: Have you started your Cal Poly SLO application?\nSam: Yes, I need to gather my transcripts and letters of recommendation.\nAlex: Remember the deadline is in two weeks.\nSam: I\xe2\x80\x99m also studying for the GRE this weekend.\nAlex: Let\xe2\x80\x99s study together tomorrow.\nSam: Sounds great!'
b'Alex and Sam discuss grad school.'


Next, we’ll group the data into batches and use only a small portion for this demo. 

In [8]:
# map each (dialogue, summary) pair to a dictionary with keys "encoder_text" and "decoder_text"
train_ds = (
    train_ds.map(
        lambda dialogue, summary: {"encoder_text": dialogue, "decoder_text": summary}
    )
    .batch(BATCH_SIZE)
    .cache()
)

# take only the first 10 batches from the dataset
train_ds = train_ds.take(NUM_BATCHES)

## Fine-tune BART

First, we load a pre‐trained BART model and a helper from KerasHub that knows how to turn raw text into the numbers the model needs. Then we create and compile the model so it’s ready to train. The helper automatically handles tokenizing  and arranging the target summary data. Once all this is done, all we have to do is feed in plain dialogue strings and the compiled model will take care of the rest


First, we’ll halve our max sequence lengths again for even faster iteration (these values are already defined above, but we’ll override here for demonstration).


In [9]:
# halve your max sequence lengths to cut attention cost drastically
MAX_ENCODER_SEQUENCE_LENGTH = 64   # was 128
MAX_DECODER_SEQUENCE_LENGTH = 32   # was 64

# load a smaller BART preset
preprocessor = keras_hub.models.BartSeq2SeqLMPreprocessor.from_preset(
    "bart_base_en",
    encoder_sequence_length=MAX_ENCODER_SEQUENCE_LENGTH,
    decoder_sequence_length=MAX_DECODER_SEQUENCE_LENGTH,
)
# load the BART Seq2Seq model
bart_lm = keras_hub.models.BartSeq2SeqLM.from_preset(
    "bart_base_en",
    preprocessor=preprocessor,
)

#print summary
bart_lm.summary()

Preprocessor: "bart_seq2_seq_lm_preprocessor"
Preprocessor: "bart_seq2_seq_lm_preprocessor"


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━
┃ Layer (type)                                                  ┃               
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━
│ bart_tokenizer (BartTokenizer)                                │               
└───────────────────────────────────────────────────────────────┴───────────────
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━
┃ Layer (type)                                                  ┃               
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━
│ bart_tokenizer (BartTokenizer)                                │               
└───────────────────────────────────────────────────────────────┴───────────────


Model: "bart_seq2_seq_lm"
Model: "bart_seq2_seq_lm"


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ 
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━
│ decoder_padding_mask          │ (None, None)              │               0 │ 
│ (InputLayer)                  │                           │                 │ 
├───────────────────────────────┼───────────────────────────┼─────────────────┼─
│ decoder_token_ids             │ (None, None)              │               0 │ 
│ (InputLayer)                  │                           │                 │ 
├───────────────────────────────┼───────────────────────────┼─────────────────┼─
│ encoder_padding_mask          │ (None, None)              │               0 │ 
│ (InputLayer)                  │                           │                 │ 
├───────────────────────────────┼───────────────────────────┼─────────────────┼─
│ encoder_token_ids         

 Total params: 139,417,344 (531.83 MB)
 Total params: 139,417,344 (531.83 MB)


 Trainable params: 139,417,344 (531.83 MB)
 Trainable params: 139,417,344 (531.83 MB)


 Non-trainable params: 0 (0.00 B)
 Non-trainable params: 0 (0.00 B)


Define the optimizer and loss. We use the Adam optimizer with a linearly
decaying learning rate. Compile the model.

In [10]:
# configure optimizer
base_optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,   # typical learning rate for fine-tuning
    weight_decay=0.01,
    epsilon=1e-6,
)
# exclude bias and layer-norm weights from weight decay
base_optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
opt = mixed_precision.LossScaleOptimizer(base_optimizer)

# define the training loss
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# compile the model
bart_lm.compile(
    optimizer=opt,
    loss=loss,
    run_eagerly=False,
)


In [11]:
#fit model with training data
bart_lm.fit(train_ds, epochs=EPOCHS)

2025-06-04 11:37:42.366599: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Epoch 1/5
Epoch 1/5


2025-06-04 11:37:44.290873: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


 1/10 ━━━━━━━━━━━━━━━━━━━━ 7:28 50s/step - loss: 1.4993 - sparse_categorical_accuracy: 0.4800

 2/10 ━━━━━━━━━━━━━━━━━━━━ 6:38 50s/step - loss: 1.2422 - sparse_categorical_accuracy: 0.5362

...

 7/10 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - loss: 0.0013 - sparse_categorical_accuracy: 0.9981

 8/10 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - loss: 0.0013 - sparse_categorical_accuracy: 0.9979

 9/10 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9978

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9978

10/10 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9978
10/10 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9978


## Generate Summaries and then Evaluate Them

Now that the model has been trained, it's time to generate summaries on the validation set and compare them to the ground‐truth human summaries.

To do this we need to:

- Define a helper function generate_text from model.generate
- Collect a list of raw dialogues from val_ds
- Perform a “warm‐up” dummy call
- Generate summaries in batches
- Print a few examples to inspect

In [12]:

def generate_text(model, input_text, max_length=200, print_time_taken=False):

    """
    Generate summaries given a batch of raw dialogue strings.
    - model: the fine-tuned BartSeq2SeqLM model
    - input_texts: either a single string or a tf.data.Dataset.batch of strings
    - max_length: maximum number of output tokens
    - print_time_taken: if True, print how many seconds the generation took
    """

    start = time.time()
    output = model.generate(input_text, max_length=max_length)
    end = time.time()
    print(f"Total Time Elapsed: {end - start:.2f}s")
    return output


In [13]:
# convert the val_ds into Python lists:
dialogues = []
ground_truth_summaries = []
for dialogue, summary in val_ds:
    dialogues.append(dialogue.numpy())
    ground_truth_summaries.append(summary.numpy())

# make a dummy call, first call usually takes a lot longer
_ = generate_text(bart_lm, "sample text", max_length=MAX_GENERATION_LENGTH)

# Ggenerate summaries for all the validation examples (there arent a lot)
generated_summaries = generate_text(
    bart_lm,
    val_ds.map(lambda dialogue, _: dialogue).batch(8),
    max_length=MAX_GENERATION_LENGTH,
    print_time_taken=True,
)

Total Time Elapsed: 6.35s
Total Time Elapsed: 6.35s


Total Time Elapsed: 7.18s
Total Time Elapsed: 7.18s


Print out a few examples that contain: the original dialogue, the original generated AI summary (via openAI), and the model’s generated summary.

In [14]:
#Print the first 3 examples with their original dialgoue, generated summary and ground truth summary.
for dialogue, generated_summary, ground_truth_summary in zip(
    dialogues[:3], generated_summaries[:3], ground_truth_summaries[:3]):

    print("Dialogue:", dialogue)
    print("Generated Summary:", generated_summary)
    print("Ground Truth Summary:", ground_truth_summary)
    print("=============================")

Dialogue: b'Liam: Olivia, are you planning to attend the data science seminar next week?\nOlivia: Yeah, it\xe2\x80\x99s being hosted by the Cal Poly ML club.\nLiam: Do you think it\xe2\x80\x99ll cover neural networks?\nOlivia: Most likely. The speaker is a researcher on CNN architectures.\nLiam: Cool, I\xe2\x80\x99ll be there.'
Generated Summary: Liv and Olivia discuss data science.
Ground Truth Summary: b'Liam asks Olivia about attending a Cal Poly seminar on data science.'
Dialogue: b'Mason: Ava, have you studied the case studies for the midterm?\nAva: I reviewed the one on supply chain analytics.\nMason: That\xe2\x80\x99s the one I was struggling with. Did you understand the KPIs?\nAva: Yeah, focus on lead time and inventory turnover.\nMason: Thanks for the clarification.'
Generated Summary: Mason and Ava discuss business analytics.
Ground Truth Summary: b'Mason and Ava discuss their business analytics midterm.'
Dialogue: b'Ethan: Sophia, want to form a study group for the ML final?

## Ouput Notes

Strengths:
- Always identifies that two students are talking about an academic topic.
- Keeps summaries very short and coherent.

Weaknesses:
- Too generic
- Loses mention of important context (like Cal Poly ML club)
- Tends to collapse everything into a broad subject rather than going into detail of it

Main Takeaway:
The model is good at flagging the broad theme, but to improve the use of this model, it needs to be able to preserve the core actions. Like attending a seminar, midterm details, or forming a study group.

## Conclusion
In this notebook, we fine-tuned a pretrained BART model on a small set of graduate-school dialogues and saw that it can learn to condense full diagloue conversations into concise summaries. Despite the limited dataset, the model captured key points within each dialogue. With more data and longer sequence lengths, we could further reduce errors and improve detail coverage. 

In practice, this sort of machine learning application could quickly generate summaries of chat logs, meeting notes, or support tickets, saving time and ensuring important information is surfaced clearly. I think there are many services out there currently that do this for services such as zoom, its very useful to get overall notes when finishing meetings.